# Perception Test Bench

Testea los modelos de IA (pose, face, hands) **sin necesidad de cámara**.

- Descarga imágenes reales de personas desde Pexels (CC0)
- Fallback a imágenes sintéticas si no hay internet
- Ejecuta cada detector y visualiza landmarks sobre la imagen
- Benchmark de latencia por modelo y en paralelo

## 1. Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import cv2
from IPython.display import display, HTML

# Paths
repo_root = Path.cwd()
if not (repo_root / "python" / "ascii_stream_engine").exists():
    repo_root = repo_root.parent.parent.parent
assert (repo_root / "python" / "ascii_stream_engine").exists(), f"Run from repo root. cwd={Path.cwd()}"

for p in [str(repo_root / "python"), str(repo_root / "cpp" / "build")]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["ONNX_MODELS_DIR"] = str(repo_root / "onnx_models" / "mediapipe")

# Verify modules
try:
    import perception_cpp
    print("perception_cpp OK")
except ImportError:
    print("ERROR: perception_cpp not found. Run: cd cpp/build && cmake .. && make")
    raise

from ascii_stream_engine.adapters.perception import (
    PoseLandmarkAnalyzer, FaceLandmarkAnalyzer, HandLandmarkAnalyzer
)
from ascii_stream_engine.application.pipeline import AnalyzerPipeline
from ascii_stream_engine.domain.config import EngineConfig

print("All imports OK")

## 2. Descargar imagenes de prueba

Descarga imagenes reales de personas desde Pexels (licencia libre).
Si falla, genera imagenes sinteticas como fallback.

In [ ]:
import urllib.request
import ssl

TEST_DIR = repo_root / "test_images"
TEST_DIR.mkdir(exist_ok=True)

# Pexels CDN - imagenes CC0 (free to use)
IMAGES = {
    "pose": {
        "file": "test_pose_real.jpg",
        "urls": [
            "https://images.pexels.com/photos/1222271/pexels-photo-1222271.jpeg?auto=compress&cs=tinysrgb&w=640",
            "https://images.pexels.com/photos/3771836/pexels-photo-3771836.jpeg?auto=compress&cs=tinysrgb&w=640",
        ],
        "desc": "Persona de cuerpo completo (pose detection)",
    },
    "face": {
        "file": "test_face_real.jpg",
        "urls": [
            "https://images.pexels.com/photos/614810/pexels-photo-614810.jpeg?auto=compress&cs=tinysrgb&w=640",
            "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=640",
        ],
        "desc": "Rostro (face detection)",
    },
    "hands": {
        "file": "test_hands_real.jpg",
        "urls": [
            "https://images.pexels.com/photos/2104252/pexels-photo-2104252.jpeg?auto=compress&cs=tinysrgb&w=640",
            "https://images.pexels.com/photos/3807517/pexels-photo-3807517.jpeg?auto=compress&cs=tinysrgb&w=640",
        ],
        "desc": "Manos (hand detection)",
    },
}

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

def download_image(name: str, info: dict) -> bool:
    """Download image, trying multiple URLs. Returns True if successful."""
    dest = TEST_DIR / info["file"]
    if dest.exists() and dest.stat().st_size > 1000:
        print(f"  {name}: ya existe ({dest.stat().st_size // 1024} KB)")
        return True
    for url in info["urls"]:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10, context=ctx) as resp:
                data = resp.read()
            if len(data) > 1000:
                dest.write_bytes(data)
                print(f"  {name}: descargada ({len(data) // 1024} KB)")
                return True
        except Exception as e:
            print(f"  {name}: fallo {url} ({e})")
    return False


def generate_synthetic(name: str, info: dict) -> None:
    """Generate synthetic fallback image."""
    dest = TEST_DIR / info["file"]
    img = np.full((480, 640, 3), 60, dtype=np.uint8)
    if name == "pose":
        cv2.circle(img, (320, 80), 35, (200, 180, 160), -1)
        cv2.rectangle(img, (290, 115), (350, 280), (180, 160, 140), -1)
        cv2.rectangle(img, (240, 130), (290, 230), (180, 160, 140), -1)
        cv2.rectangle(img, (350, 130), (400, 230), (180, 160, 140), -1)
        cv2.rectangle(img, (300, 280), (320, 420), (160, 140, 120), -1)
        cv2.rectangle(img, (330, 280), (350, 420), (160, 140, 120), -1)
    elif name == "face":
        cv2.ellipse(img, (320, 240), (100, 130), 0, 0, 360, (200, 180, 160), -1)
        cv2.circle(img, (285, 210), 12, (40, 40, 40), -1)
        cv2.circle(img, (355, 210), 12, (40, 40, 40), -1)
        cv2.ellipse(img, (320, 280), (35, 15), 0, 0, 180, (120, 80, 80), 3)
    elif name == "hands":
        cv2.ellipse(img, (200, 260), (55, 75), 20, 0, 360, (200, 180, 160), -1)
        for off in [-28, -14, 0, 14, 28]:
            cv2.ellipse(img, (200 + off, 190), (7, 35), 0, 0, 360, (200, 180, 160), -1)
        cv2.ellipse(img, (440, 260), (55, 75), -20, 0, 360, (200, 180, 160), -1)
        for off in [-28, -14, 0, 14, 28]:
            cv2.ellipse(img, (440 + off, 190), (7, 35), 0, 0, 360, (200, 180, 160), -1)
    cv2.imwrite(str(dest), img)
    print(f"  {name}: sintetica generada (fallback)")


print("Descargando imagenes de prueba...")
for name, info in IMAGES.items():
    if not download_image(name, info):
        generate_synthetic(name, info)

print("\nImagenes listas.")

## 3. Cargar imagenes y visualizar

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

loaded = {}
for name, info in IMAGES.items():
    path = TEST_DIR / info["file"]
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        print(f"ERROR: no se pudo cargar {path}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    loaded[name] = img_rgb
    print(f"{name}: {img_rgb.shape} loaded")

fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 5))
if len(loaded) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, loaded.items()):
    ax.imshow(img)
    ax.set_title(f"{name} ({img.shape[1]}x{img.shape[0]})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Deteccion individual por modelo

Ejecuta cada detector sobre su imagen correspondiente y muestra los landmarks.

In [ ]:
def draw_landmarks(img_rgb, landmarks, color=(0, 255, 0), radius=4):
    """Draw landmarks on a copy of the image. landmarks: (N,2) normalized 0-1."""
    vis = img_rgb.copy()
    h, w = vis.shape[:2]
    for i in range(landmarks.shape[0]):
        x = int(landmarks[i, 0] * w)
        y = int(landmarks[i, 1] * h)
        if 0 <= x < w and 0 <= y < h:
            cv2.circle(vis, (x, y), radius, color, -1)
            cv2.circle(vis, (x, y), radius + 1, (255, 255, 255), 1)
    return vis


detectors = {
    "pose": ("detect_pose", (0, 255, 0)),
    "face": ("detect_face", (255, 100, 100)),
    "hands": ("detect_hands", (100, 100, 255)),
}

results = {}
fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 5))
if len(loaded) == 1:
    axes = [axes]

for ax, (name, img) in zip(axes, loaded.items()):
    func_name, color = detectors.get(name, (None, (0, 255, 0)))
    if func_name is None or not hasattr(perception_cpp, func_name):
        ax.imshow(img)
        ax.set_title(f"{name}: no detector")
        ax.axis("off")
        continue

    detect_fn = getattr(perception_cpp, func_name)

    # Warm up
    detect_fn(img)

    # Timed run
    t0 = time.perf_counter()
    raw = detect_fn(img)
    dt = (time.perf_counter() - t0) * 1000

    n_pts = raw.shape[0] if raw is not None and raw.size > 0 else 0
    results[name] = {"raw": raw, "points": n_pts, "time_ms": dt}

    if n_pts > 0:
        # Normalize if not already in 0-1 range
        h, w = img.shape[:2]
        lm = raw.copy().astype(np.float32)
        if lm.max() > 1.5:  # pixel coords, normalize
            lm[:, 0] /= w
            lm[:, 1] /= h
        vis = draw_landmarks(img, lm, color)
        ax.imshow(vis)
    else:
        ax.imshow(img)

    ax.set_title(f"{name}: {n_pts} pts ({dt:.1f} ms)")
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\nResumen:")
for name, r in results.items():
    print(f"  {name}: {r['points']} puntos, {r['time_ms']:.1f} ms")

## 5. AnalyzerPipeline completo (secuencial vs paralelo)

In [ ]:
cfg = EngineConfig()

# Usar la imagen de pose para todos los analyzers
test_frame = loaded.get("pose", loaded.get("face", list(loaded.values())[0]))

# -- Sequential benchmark (1 analyzer at a time) --
analyzers_list = [
    PoseLandmarkAnalyzer(),
    FaceLandmarkAnalyzer(),
    HandLandmarkAnalyzer(),
]

# Warmup
for a in analyzers_list:
    a.analyze(test_frame, cfg)

t0 = time.perf_counter()
sequential_results = {}
for a in analyzers_list:
    name = a.name
    sequential_results[name] = a.analyze(test_frame, cfg)
t_seq = (time.perf_counter() - t0) * 1000

print(f"Sequential (3 analyzers): {t_seq:.1f} ms")
for name, data in sequential_results.items():
    if isinstance(data, dict):
        for k, v in data.items():
            pts = v.shape[0] if hasattr(v, 'shape') and v.ndim > 0 else 0
            print(f"  {name}.{k}: {pts} pts")
    else:
        print(f"  {name}: {data}")

# -- Parallel benchmark (AnalyzerPipeline) --
pipeline = AnalyzerPipeline([
    PoseLandmarkAnalyzer(),
    FaceLandmarkAnalyzer(),
    HandLandmarkAnalyzer(),
])

# Warmup
pipeline.run(test_frame, cfg)

N = 20
t0 = time.perf_counter()
for _ in range(N):
    parallel_results = pipeline.run(test_frame, cfg)
t_par = ((time.perf_counter() - t0) / N) * 1000

print(f"\nParallel (3 analyzers, avg of {N}): {t_par:.1f} ms")
for name, data in parallel_results.items():
    if isinstance(data, dict):
        for k, v in data.items():
            pts = v.shape[0] if hasattr(v, 'shape') and v.ndim > 0 else 0
            print(f"  {name}.{k}: {pts} pts")

if t_seq > 0:
    print(f"\nSpeedup: {t_seq / t_par:.2f}x")

## 6. Benchmark de latencia por modelo

Mide latencia promedio/min/max de cada modelo sobre 50 iteraciones.

In [ ]:
BENCH_N = 50
bench_frame = loaded.get("pose", list(loaded.values())[0])

for func_name, label in [("detect_pose", "Pose"), ("detect_face", "Face"), ("detect_hands", "Hands")]:
    fn = getattr(perception_cpp, func_name)
    fn(bench_frame)  # warmup

    times = []
    for _ in range(BENCH_N):
        t0 = time.perf_counter()
        fn(bench_frame)
        times.append((time.perf_counter() - t0) * 1000)

    arr = np.array(times)
    print(f"{label:6s}: avg={arr.mean():.1f} ms | min={arr.min():.1f} ms | max={arr.max():.1f} ms | std={arr.std():.1f} ms")

## 7. Test con imagen propia (opcional)

Carga tu propia imagen para probar los detectores.

In [ ]:
# Cambia esta ruta a tu imagen:
CUSTOM_IMAGE = None  # Ejemplo: "/home/user/mi_foto.jpg"

if CUSTOM_IMAGE and Path(CUSTOM_IMAGE).exists():
    custom_bgr = cv2.imread(CUSTOM_IMAGE)
    custom_rgb = cv2.cvtColor(custom_bgr, cv2.COLOR_BGR2RGB)
    print(f"Imagen cargada: {custom_rgb.shape}")

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(custom_rgb)
    axes[0].set_title("Original")
    axes[0].axis("off")

    h, w = custom_rgb.shape[:2]
    for i, (func_name, label, color) in enumerate([
        ("detect_pose", "Pose", (0, 255, 0)),
        ("detect_face", "Face", (255, 100, 100)),
        ("detect_hands", "Hands", (100, 100, 255)),
    ]):
        raw = getattr(perception_cpp, func_name)(custom_rgb)
        n_pts = raw.shape[0] if raw is not None and raw.size > 0 else 0
        if n_pts > 0:
            lm = raw.copy().astype(np.float32)
            if lm.max() > 1.5:
                lm[:, 0] /= w
                lm[:, 1] /= h
            vis = draw_landmarks(custom_rgb, lm, color)
            axes[i + 1].imshow(vis)
        else:
            axes[i + 1].imshow(custom_rgb)
        axes[i + 1].set_title(f"{label}: {n_pts} pts")
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Sin imagen custom. Edita CUSTOM_IMAGE para probar con tu propia foto.")